# 1. 数据集的结构编码

## 1.1 子图构建

对于过大的图（比如 Cora、Citeseer 这种只有一个图），我们先将其拆分为一系列子图（暂时定为最多不超过 64 个节点一个图）：

$$
G \rightarrow g_1, g_2, \ldots, g_n
$$

## 1.2 结构编码的定义

对于每个子图 $g_i$，我们使用结构统计特征来构造其结构编码：

$$
s_i = \left(|V_i|,\ |E_i|,\ \bar{d}_i,\ d_i^{\max},\ \mathrm{Var}(d_i),\ \mathrm{Density}(g_i),\ \mathrm{Clust}(g_i),\ \mathrm{Tri}(g_i),\ \mathrm{Diam}(g_i)\right)
$$

其中：



## 1.3 基于 KMeans 的原型划分

我们对所有结构编码执行 KMeans 聚类：

$$
S \xrightarrow{\mathrm{KMeans}(K)} P_1, P_2, \ldots, P_K
$$

其中：

- $K$ 是超参数；
- 每个簇对应一种典型的结构模式；
- 第 $k$ 个簇的中心记为 $p_k$。

这些簇中心组成集合：

$$
P = p_1, p_2, \ldots, p_K
$$

它们用于刻画整个数据集的全局结构分布。

# 2. 可学分数

我们为每个样本分配一个可学分数，用来衡量它是否值得被用于预训练。该分数由三部分组成：**Novelty、Coverage 和 Redundancy**。

## 2.1 代表池

在得到聚类结果后，我们从每个簇中选取若干个最具代表性的样本，构成代表池。（这些代表池里的样本将直接进入训练中）

对于第 $k$ 个簇，我们选取与簇中心 $p_k$ 最接近的前 $m$ 个样本：

$$
S_k = \mathrm{TopMClosestTo}(p_k)
$$

然后将所有簇中的代表样本合并，得到全局代表池：

$$
S = \bigcup_{k=1}^{K} S_k
$$

其中，$S = r_1, r_2, \ldots, r_M$，每个 $r_j$ 都是一个代表性结构向量，其实也就是一个 $s_k$。

## 2.2 Novelty 分数

Novelty 用于衡量一个新样本是否能够在当前代表池之外提供新的结构信息。

我们定义一个覆盖函数：

$$
f(S) = \sum_{i \in M} \max_{r_j \in S} \cos(s_i, r_j)
$$

其中，$M, S$ 为两个集合。

对于一个节点 $v_i$ 而言，代表池对它的覆盖是一定的：

$$
c_{j0} = \max_{r_j \in S} \cos(s_i, r_j)
$$

一个样本的 Novelty 定义为其加入代表池后所带来的覆盖增益：

$$
\mathrm{Novelty}(s_i) = f(\{S \cup s_i\}) - f(S)
= \sum_{j \in M} \max(\cos(s_i, s_j) - c_{j0}, 0)
$$

注意这里简化的部分是由于代表池在每个点内的覆盖是个常数，所以事实上只需要算新点对其余点的覆盖即可。

该值越大，说明该样本能够补充当前代表池尚未充分覆盖的结构区域。

## 2.3 Coverage 分数

Coverage 用于强调稀缺结构模式的重要性。

设样本 $s_i$ 被分配到第 $c_i$ 个簇，该簇在整个数据集中的出现概率为：

$$
p_i = \frac{|P_{c_i}|}{N}
$$

则 Coverage 分数定义为：

$$
\mathrm{Coverage}(s_i) = -\log p_i
$$

因此，数据集中较少出现的结构模式会被赋予更高的学习优先级。

## 2.4 Redundancy 分数

Redundancy 用于衡量一个样本是否与当前固定代表池过于相似：

$$
\mathrm{Redundancy}(s_i) = - \max_{r_j \in S} \cos(s_i, r_j)
$$

Redundancy 越大，说明该样本与已有代表越重复，其额外学习价值越低。

## 2.5 可学分数

三项可学分数 Novelty、Coverage 和 Redundancy 先做归一化（比如 Max-min）转化为 0～1 之间的值。

最终的可学分数由三项组合得到：

$$
\mathrm{Learnability}(s_i)
= \alpha \cdot \mathrm{Novelty}(s_i)
+ \beta \cdot \mathrm{Coverage}(s_i)
+ \gamma \cdot \mathrm{Redundancy}(s_i)
$$

# 3. 基于结构特征的对比学习EGRAPHCL

传统对比学习通常采用正负例配对方式：在同一样本的两种增强视图被视为正例，其余所有样本均被视为负例。但是由于不同样本之间仍然可能共享较强的结构语义或拓扑模式。如果将这类样本强行作为负例，会引入错误负例，从而损害表示学习效果。

为此，我们将样本对划分为三类：**强正例、软正例和真负例**。

## 3.1 强正例：自身增强对

对于同一样本 $g_i$ 的两种增强视图 $g_i^{(1)}$ 和 $g_i^{(2)}$，我们将其视为强正例，即：

$$
(g_i^{(1)}, g_i^{(2)}) \in \mathcal{P}_{\text{strong}}
$$

## 3.2 软正例：高相似结构样本对

对于两个不同样本 $g_i$ 和 $g_j$，若它们在结构编码空间中具有较高相似度，则说明它们共享较强的结构语义，不应被简单视为负例。

设对应的结构编码为 $s_i$ 和 $s_j$，定义它们的语义相似度为：

$$
w_{ij} = \cos(s_i, s_j)
$$

当相似度高于阈值 $\tau$ 时，我们将该样本对视为软正例：

$$
(g_i, g_j) \in \mathcal{P}_{\text{soft}}
\quad \text{if} \quad
w_{ij} > \tau
$$

目前我们先当它不存在，也就是不参与 InfoNCE 的 loss 计算。

## 3.3 真负例：低共享语义样本对

对于两个不同样本 $g_i$ 和 $g_j$，若它们在结构编码空间中的相似度较低，则说明它们在结构模式上差异较大，可以视为真正应被区分开的负例。

当相似度低于阈值 $\tau$ 时，我们将该样本对视为真负例：

$$
(g_i, g_j) \in \mathcal{P}_{\text{neg}}
\quad \text{if} \quad
w_{ij} < \tau
$$

这类样本对用于维持表示空间中的判别性。

## 3.5 三分类配对策略总结

因此，我们的对比学习配对方式可以总结为：

- **强正例**：同一样本的两种增强视图；
- **软正例**：不同样本中具有高共享语义的样本对（不参与训练）；
- **真负例**：不同样本中共享语义较低的样本对。

形式化地，可写为：

$$
\begin{cases}
(g_i^{(1)}, g_i^{(2)}) \in \mathcal{P}_{\text{strong}}, \\\\
(g_i, g_j) \in \mathcal{P}_{\text{soft}}, & w_{ij} > \tau, \\\\
(g_i, g_j) \in \mathcal{P}_{\text{neg}}, & w_{ij} < \tau.
\end{cases}
$$